# Qudit cluster chain 50 site DMRG

Created 05/03/2026

Calculate a qudit cluster chain MPS with 50 sites, $d=7$.

In [1]:
import numpy as np
import scipy
import matplotlib.pyplot as plt

In [2]:
import tenpy
import tenpy.linalg.np_conserved as npc
from tenpy.algorithms import dmrg
from tenpy.networks.mps import MPS

In [3]:
from tenpy.networks.site import SpinSite, ClockSite
from tenpy.models.lattice import Chain
from tenpy.models.model import CouplingModel, NearestNeighborModel, MPOModel

In [34]:
class QuditClusterChain(CouplingModel):
    def __init__(self, model_params):
        # Read out/set default parameters
        name = "Cluster Ising model"
        L = model_params.get('L', 5)
        d = model_params.get('d', 7)
        bc_MPS = model_params.get('bc_MPS', 'finite')
        
        # sites
        site = ClockSite(d, conserve=None)

        # lattice
        bc = 'open'
        lat = Chain(L, site, bc=bc, bc_MPS=bc_MPS)

        # initialize CouplingModel
        CouplingModel.__init__(self, lat)

        # add terms of the Hamiltonian
        for i in range(1, d): 
            self.add_multi_coupling(
                -1/d,
                [
                    *([('Zhc', -1, 0)]*i),
                    *([('X', 0, 0)]*i),
                    *([('Z', 1, 0)]*i),
                ]
            )

        # initialize H_MPO
        MPOModel.__init__(self, lat, self.calc_H_MPO())

In [76]:
class QuditClusterChain2(CouplingModel):
    def __init__(self, model_params):
        # Read out/set default parameters
        name = "Cluster Ising model"
        L = model_params.get('L', 5)
        d = model_params.get('d', 7)
        bc_MPS = model_params.get('bc_MPS', 'finite')
        
        # sites
        site = ClockSite(d, conserve=None)

        # lattice
        bc = 'open'
        lat = Chain(L, site, bc=bc, bc_MPS=bc_MPS)

        # initialize CouplingModel
        CouplingModel.__init__(self, lat)

        # add terms of the Hamiltonian
        self.add_multi_coupling(
            -1,
            [
                ('Zhc', -1, 0),
                ('X', 0, 0),
                ('Z', 1, 0),
            ]
        )

        self.add_multi_coupling(
            -1,
            [
                ('Z', -1, 0),
                ('Xhc', 0, 0),
                ('Zhc', 1, 0),
            ]
        )

        # initialize H_MPO
        MPOModel.__init__(self, lat, self.calc_H_MPO())

In [6]:
d=7

In [37]:
dmrg_params = {
    "trunc_params": {"chi_max": 8, "chi_min": d, "svd_min": 1.e-10},
    "min_sweeps":10,
    "max_sweeps":20,
    "mixer": True,
    "combine":False,
    'decay':2,
    'amplitude':10e-1,
    'disable_after':60,
    'update_env':0
}

In [8]:
import h5py
from tenpy.tools import hdf5_io

In [77]:
d=3

In [78]:
dmrg_params = {
    "trunc_params": {"chi_max": d, "chi_min": 1},
    "min_sweeps":10,
    "max_sweeps":20,
    "mixer": True,
    "combine":False,
    'decay':2,
    'amplitude':10e-1,
    'disable_after':60,
    'update_env':0
}

In [79]:
model=QuditClusterChain2({'d': d, 'L':30})

psi = MPS.from_desired_bond_dimension(
    model.lat.mps_sites(),
    d,
    bc=model.lat.bc_MPS
)
psi.canonical_form()

In [80]:
(
    model
    .all_coupling_terms()
    .to_TermList()
    .terms
)

[[('Zhc', 0), ('X', 1), ('Z', 2)],
 [('Zhc', 1), ('X', 2), ('Z', 3)],
 [('Zhc', 2), ('X', 3), ('Z', 4)],
 [('Zhc', 3), ('X', 4), ('Z', 5)],
 [('Zhc', 4), ('X', 5), ('Z', 6)],
 [('Zhc', 5), ('X', 6), ('Z', 7)],
 [('Zhc', 6), ('X', 7), ('Z', 8)],
 [('Zhc', 7), ('X', 8), ('Z', 9)],
 [('Zhc', 8), ('X', 9), ('Z', 10)],
 [('Zhc', 9), ('X', 10), ('Z', 11)],
 [('Zhc', 10), ('X', 11), ('Z', 12)],
 [('Zhc', 11), ('X', 12), ('Z', 13)],
 [('Zhc', 12), ('X', 13), ('Z', 14)],
 [('Zhc', 13), ('X', 14), ('Z', 15)],
 [('Zhc', 14), ('X', 15), ('Z', 16)],
 [('Zhc', 15), ('X', 16), ('Z', 17)],
 [('Zhc', 16), ('X', 17), ('Z', 18)],
 [('Zhc', 17), ('X', 18), ('Z', 19)],
 [('Zhc', 18), ('X', 19), ('Z', 20)],
 [('Zhc', 19), ('X', 20), ('Z', 21)],
 [('Zhc', 20), ('X', 21), ('Z', 22)],
 [('Zhc', 21), ('X', 22), ('Z', 23)],
 [('Zhc', 22), ('X', 23), ('Z', 24)],
 [('Zhc', 23), ('X', 24), ('Z', 25)],
 [('Zhc', 24), ('X', 25), ('Z', 26)],
 [('Zhc', 25), ('X', 26), ('Z', 27)],
 [('Zhc', 26), ('X', 27), ('Z', 28)],
 

In [85]:
d=3

In [86]:
dmrg_params = {
    "mixer": True,
    "max_E_err": 1e-10,
    "trunc_params": {
        "chi_max": 2*d,
        "svd_min": 1e-10
    },
    "max_sweeps": 10
}

In [87]:
model=QuditClusterChain2({'d': d, 'L':30})

psi = MPS.from_desired_bond_dimension(
    model.lat.mps_sites(),
    d,
    bc=model.lat.bc_MPS
)
psi.canonical_form()

In [88]:
eng = dmrg.TwoSiteDMRGEngine(psi, model, dmrg_params)
e, psi = eng.run()

print(e)

-34.65420387078656


/Users/kierancooney/.pyenv/versions/num_spt_venv_p11/lib/python3.11/site-packages/tenpy/algorithms/mps_common.py:783: TenpyInconsistencyWarning: Maximum truncation error (``max_trunc_err``) exceeded.
  consistency_check(np.max(self.trunc_err_list), self.options, 'max_trunc_err', 1e-4,


Bond dimensions look ok...

In [18]:
filename = r"../data/qudit_cluster_chain_d_7_L_30.h5"
with h5py.File(filename, 'w') as f:
    hdf5_io.save_to_hdf5(f, psi)